# Lean-19b : Borne de Chernoff pour le BSC -- extension de Lean-19

**Navigation** : [<< Lean-19 Hoeffding](Lean-19-Detection-MIMO-Index-Python.ipynb) · [Lean-19c Bernstein](Lean-19c-Borne-Bernstein-CSharp.ipynb) · [Lean-19d Bayésien](Lean-19d-Inference-Bayesian-Beta-Binomial-Python.ipynb) · [Version Python](Lean-19b-Borne-Chernoff-Python.ipynb) · [Sommaire série](README.md)
Carnet d'**extension methodologique** de Lean-19 (Hoeffding) : derivation de la borne de
Chernoff pour le canal binaire symetrique (BSC), comparaison rigoureuse avec Hoeffding
(distribution-free) et Bernstein (intermediaire) sur 20 instances BSC.

**Enonce** : Pour une somme de N variables iid Bernoulli(p), on derive la borne de
Chernoff en exploitant la fonction generatrice des moments (mgf) exacte de chaque Bernoulli X :
```
M_X(t) = E[e^(tX)] = (1 - p) + p * e^t
```
L'expression de Chernoff dans l'approximation KL quadratique est :
```
2 * exp(-N * eps^2 / (2 p (1-p)))
```
Elle approxime l'expression bilatérale de Chernoff obtenue avec les divergences KL des deux queues ; elle n'est pas une majoration garantie à écart fini. La borne de Hoeffding (distribution-free) suppose le pire cas p=1/2 :
```
P(|p_hat - p| >= eps) <= 2 * exp(-2 N eps^2)
```

**Pourquoi ce carnet est un sous-carnet de Lean-19** : Lean-19 a etabli empiriquement que
la detection MIMO tient avec la borne de Hoeffding sur 20 instances. Lean-19b montre
que la **borne de Chernoff est plus tight** pour la distribution Bernoulli, et fournit
l'outil de reference pour la detection MIMO quand la distribution est connue.

Protocole de mesure : pour chaque instance, le canal est simulé `n_runs` fois sur `t` pas — à chaque pas, un flip a lieu avec probabilité `p` (`rng.NextDouble() < p`) et l'état avance modulo `M`. On obtient `N = n_runs * t` tirages, la fréquence observée `p_hat = n_flips / N`, et trois rayons `eps` au niveau nominal de confiance 95 % (`eps_p = 0.05`) : Hoeffding (distribution-free, pire cas `p(1-p) = 1/4`), Chernoff (approximation KL quadratique, vraie variance `p(1-p)`) et Bernstein (avec le terme de correction `eps/3`).

À niveau de confiance fixé, le plus petit `eps` est le rayon le plus serré (tight). Les tableaux imprimés comparent donc systématiquement `hoeff(H)`, `chern(C)` et `bernst(B)` ligne par ligne.

In [1]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Graine pour reproductibilite (meme seed que Lean-19)
const int seed = 42;
var rng = new Random(seed);
const double epsP = 0.05;

// Tolérance absolue sur les formules fermées, avant arrondi d'affichage.
const double boundTolerance = 1e-12;
// Trois erreurs-types de la moyenne de N tirages de Bernoulli indépendants.
const double monteCarloSigmaMultiplier = 3.0;

string Inv(FormattableString text) => text.ToString(CultureInfo.InvariantCulture);

void Require(bool condition, string message)
{
    if (!condition)
        throw new InvalidOperationException(message);
}

Require(seed == 42, "Graine différente de la référence.");
Require(epsP == 0.05 && Inv($"{1.0 - epsP:F2}") == "0.95", "Niveau de confiance différent de la référence.");
Console.WriteLine(Inv($"Lean-19b seed={seed} (rng .NET System.Random, continuation de Lean-19)"));

Lean-19b seed=42 (rng .NET System.Random, continuation de Lean-19)


### Lecture de la sortie : la graine est fixée

La sortie affiche `Lean-19b seed=42 (rng .NET System.Random, continuation de Lean-19)` : le générateur pseudo-aléatoire est ancré sur la graine 42, comme dans Lean-19, donc réexécuter le carnet avec le même environnement .NET rejoue exactement les mêmes tirages et produit les mêmes tableaux — les valeurs citées dans les lectures suivantes sont reproductibles à l'identique.

### Note de parité : mêmes bornes, nouveaux tirages

Les tableaux de bornes coïncident à la précision affichée avec les sorties Python citées ; avant arrondi, les trois formules fermées sont contrôlées à `1e-12` par un calcul décimal indépendant sur les mêmes entrées. Les flips et `p_hat` sont recalculés avec `new Random(42)` et varient d'un générateur à l'autre : ce ne sont pas les tirages du PCG64 de NumPy.

Pour chaque instance, on affiche et contrôle `|p_hat - p|` avec la tolérance fixée avant simulation : `3 * sqrt(p*(1-p)/N)`, soit trois erreurs-types de la fréquence de Bernoulli. Ce seuil est un contrôle Monte-Carlo par instance, pas une garantie de couverture simultanée. On vérifie séparément l'invariant empirique de référence : 12/12 puis 8/8, soit 20/20 observations sous le rayon de Hoeffding.

In [2]:
// Illustration 1 — MGF de Bernoulli et comparaison sur 12 instances

double HoeffdingEps(int n, double p, double eps_p = epsP)
{
    // Borne de Hoeffding (two-sided, distribution-free, range [0,1]).
    // Forme standard : P(|p_hat - p| >= eps) <= 2 * exp(-2 * N * eps^2).
    return Math.Sqrt(Math.Log(2.0 / eps_p) / (2.0 * n));
}

double ChernoffEps(int n, double p, double eps_p = epsP)
{
    if (p == 0.0 || p == 1.0)
        return 0.0;
    return Math.Sqrt(2.0 * Math.Log(2.0 / eps_p) * p * (1.0 - p) / n);
}

double BernsteinEps(int n, double p, double eps_p = epsP)
{
    // Borne de Bernstein (two-sided, range [0,1]).
    // Forme standard : P(|p_hat - p| >= eps) <= 2 * exp(-N * eps^2 / (2*(sigma^2 + eps/3))).
    // Solution fermee : eps = ((2/3)*L + sqrt((2/3)^2 L^2 + 8*N*L*sigma^2)) / (2*N),
    // avec L = ln(2/eps_p) et sigma^2 = p*(1-p).
    if (p == 0.0 || p == 1.0)
        return HoeffdingEps(n, p, eps_p);
    double l = Math.Log(2.0 / eps_p);
    double sigma2 = p * (1.0 - p);
    double a = (2.0 / 3.0) * l;
    double disc = a * a + 8.0 * n * l * sigma2;
    return (a + Math.Sqrt(disc)) / (2.0 * n);
}

// Vérification indépendante en décimal : les sorties Python ne publient que cinq décimales.
// On utilise ln(40) évalué à 28 décimales et la racine carrée par itérations de Newton.
decimal DecimalSqrt(decimal value)
{
    if (value == 0m)
        return 0m;
    decimal x = value >= 1m ? value : 1m;
    for (int i = 0; i < 32; i++)
        x = (x + value / x) / 2m;
    return x;
}

(double H, double C, double B) BoundsReference(int n, decimal p)
{
    const decimal log40 = 3.6888794541139363028524556976m;
    decimal variance = p * (1m - p);
    decimal h = DecimalSqrt(log40 / (2m * n));
    decimal c = h * 2m * DecimalSqrt(variance);
    decimal a = log40 / (3m * n);
    decimal b = a + DecimalSqrt(a * a + 2m * log40 * variance / n);
    return ((double)h, (double)c, (double)b);
}

double VerifyBound(string column, string instance, double value, double reference, string pythonDisplay)
{
    double delta = Math.Abs(value - reference);
    Require(delta <= boundTolerance,
        Inv($"{instance}, {column} : valeur {value:G17}, référence décimale {reference:G17}, écart {delta:E3} > {boundTolerance:E1}."));
    string actualDisplay = Inv($"{value:F5}");
    Require(actualDisplay == pythonDisplay,
        $"{instance}, {column} : affichage C# {actualDisplay}, sortie Python citée {pythonDisplay}.");
    return delta;
}

(int M, int T, double P, int N, int Flips, double PHat, double H, double C, double B,
    double Error, double McLimit, double DeltaH, double DeltaC, double DeltaB)
Measure(int m, int t, double p, int runs,
    (int M, int T, decimal P, int N, string H, string C, string B) expected)
{
    int n = runs * t;
    string instance = Inv($"M={m}, t={t}, p={p:F3}, N={n}");
    Require(m == expected.M && t == expected.T && n == expected.N
        && Math.Abs(p - (double)expected.P) <= boundTolerance
        && Inv($"{p:F3}") == Inv($"{expected.P:F3}"),
        $"Entrées différentes de la référence Python : {instance}.");

    double mcLimit = monteCarloSigmaMultiplier * Math.Sqrt(p * (1.0 - p) / n);
    int flips = 0;
    for (int run = 0; run < runs; run++)
    {
        int state = rng.Next(0, m);
        for (int step = 0; step < t; step++)
        {
            if (rng.NextDouble() < p)
            {
                state = (state + 1) % m;
                flips++;
            }
        }
    }

    double pHat = (double)flips / n;
    double error = Math.Abs(pHat - p);
    double h = HoeffdingEps(n, p);
    double c = ChernoffEps(n, p);
    double b = BernsteinEps(n, p);
    var reference = BoundsReference(expected.N, expected.P);
    double deltaH = VerifyBound("hoeff(H)", instance, h, reference.H, expected.H);
    double deltaC = VerifyBound("chern(C)", instance, c, reference.C, expected.C);
    double deltaB = VerifyBound("bernst(B)", instance, b, reference.B, expected.B);
    Require(error <= mcLimit, Inv($"{instance} : |p_hat - p| = {error:G17} > seuil Monte-Carlo {mcLimit:G17}."));
    Require(expected.P == 0.50m ? c == h : c < h,
        Inv($"{instance} : ordre C/H différent de la référence (C={c:G17}, H={h:G17})."));
    return (m, t, p, n, flips, pHat, h, c, b, error, mcLimit, deltaH, deltaC, deltaB);
}

void PrintTable((int M, int T, double P, int N, int Flips, double PHat, double H, double C, double B,
    double Error, double McLimit, double DeltaH, double DeltaC, double DeltaB)[] instances, int expectedCovered)
{
    Require(instances.Length == expectedCovered, "Effectif du lot différent de la référence.");
    Console.WriteLine("   M   t      p      N   p_hat  hoeff(H)  chern(C) bernst(B)   flips  |p_hat-p|    3 sigma  Contrôle");
    foreach (var r in instances)
        Console.WriteLine(Inv($"{r.M,4} {r.T,3} {r.P,6:F3} {r.N,6} {r.PHat,7:F4} {r.H,9:F5} {r.C,9:F5} {r.B,9:F5} {r.Flips,7} {r.Error,10:F6} {r.McLimit,10:F6}  CONFORME"));

    int covered = instances.Count(r => r.Error <= r.H);
    Require(covered == expectedCovered,
        Inv($"Couverture Hoeffding : {covered}/{instances.Length} ; attendu {expectedCovered}/{instances.Length}."));
    Console.WriteLine(Inv($"Sous Hoeffding : {covered}/{instances.Length} (attendu : {expectedCovered}/{instances.Length})."));
    Console.WriteLine(Inv($"Écarts maximaux aux références décimales : H={instances.Max(r => r.DeltaH):E3}, C={instances.Max(r => r.DeltaC):E3}, B={instances.Max(r => r.DeltaB):E3} (tolérance {boundTolerance:E1})."));
}

// 12 instances de reference (meme seed que Lean-19)
// Entrées et affichages des bornes cités de la sortie Python, cellule 3 ; aucun tirage cité.
var pythonRef = new (int M, int T, decimal P, int N, string H, string C, string B)[]
{
    ( 4, 3, 0.100m, 3000, "0.02480", "0.01488", "0.01529"),
    ( 5, 4, 0.150m, 4000, "0.02147", "0.01534", "0.01565"),
    ( 6, 5, 0.200m, 5000, "0.01921", "0.01537", "0.01561"),
    ( 7, 3, 0.250m, 3000, "0.02480", "0.02147", "0.02189"),
    ( 8, 4, 0.300m, 4000, "0.02147", "0.01968", "0.01999"),
    ( 9, 5, 0.350m, 5000, "0.01921", "0.01832", "0.01857"),
    (10, 3, 0.400m, 3000, "0.02480", "0.02429", "0.02471"),
    (11, 4, 0.450m, 4000, "0.02147", "0.02137", "0.02168"),
    (12, 5, 0.500m, 5000, "0.01921", "0.01921", "0.01945"),
    (13, 3, 0.550m, 3000, "0.02480", "0.02467", "0.02508"),
    (14, 4, 0.600m, 4000, "0.02147", "0.02104", "0.02135"),
    (15, 5, 0.650m, 5000, "0.01921", "0.01832", "0.01857")
};
const int referenceCount = 12;
const int referenceRuns = 1000;
Require(pythonRef.Length == referenceCount, "La référence Python doit comporter les douze instances.");
var instancesRef = Enumerable.Range(0, referenceCount)
    .Select(i => Measure(4 + i, 3 + i % 3, 0.10 + 0.05 * i, referenceRuns, pythonRef[i]))
    .ToArray();
Require(instancesRef.Count(r => r.C < r.H) == 11 && instancesRef.Count(r => r.C == r.H) == 1,
    "Le lot de référence doit donner onze inégalités strictes et une égalité.");

Console.WriteLine("=== Cellule 1 : mgf Bernoulli et borne de Chernoff ===");
Console.WriteLine("Pour X ~ Bernoulli(p), la fonction generatrice des moments est :");
Console.WriteLine("  M_X(t) = E[e^(tX)] = (1 - p) + p * e^t");
Console.WriteLine();
Console.WriteLine("Expression bilatérale de Chernoff dans l'approximation KL quadratique :");
Console.WriteLine("  2 * exp(-N * eps^2 / (2 p (1-p)))");
Console.WriteLine();
Console.WriteLine("Borne de Hoeffding (two-sided, distribution-free) :");
Console.WriteLine("  P(|p_hat - p| >= eps) <= 2 * exp(-2 N eps^2)");
Console.WriteLine();
Console.WriteLine(Inv($"Pour un meme niveau de confiance nominal 1 - eps_p = {1.0 - epsP:F2} :"));
PrintTable(instancesRef, referenceCount);

=== Cellule 1 : mgf Bernoulli et borne de Chernoff ===


Pour X ~ Bernoulli(p), la fonction generatrice des moments est :


  M_X(t) = E[e^(tX)] = (1 - p) + p * e^t


Expression bilatérale de Chernoff dans l'approximation KL quadratique :


  2 * exp(-N * eps^2 / (2 p (1-p)))


Borne de Hoeffding (two-sided, distribution-free) :


... (13 outputs left out)

  14   4  0.600   4000  0.5897   0.02147   0.02104   0.02135    2359   0.010250   0.023238  CONFORME


  15   5  0.650   5000  0.6424   0.01921   0.01832   0.01857    3212   0.007600   0.020236  CONFORME


Sous Hoeffding : 12/12 (attendu : 12/12).


Écarts maximaux aux références décimales : H=3.469E-018, C=3.469E-018, B=3.469E-018 (tolérance 1.0E-012).


### Lecture du résultat : Chernoff sous Hoeffding sur toutes les lignes sauf une

Sur la première instance (`M=4, t=3, p=0.100, N=3000`), le tirage .NET donne `p_hat = 0.1090`, soit `|p_hat - p| = 0.009000`. La colonne `chern(C)` est strictement inférieure à `hoeff(H)` sur toutes les lignes sauf une : dès la première, `eps_C = 0.01488` contre `eps_H = 0.02480`. Seule la ligne `p = 0.500` (`M=12, N=5000`) affiche l'égalité numérique `eps_H = eps_C = 0.01921` : à `p = 1/2`, `2*p*(1-p) = 1/2` et l'exposant de Chernoff rejoint celui de Hoeffding, les deux expressions comparées sont alors identiques.

## Mesure 2 : extension sur 8 nouvelles instances

Les 12 instances precedentes varient `M` (4 a 15) et `t` (3 a 5) avec `p` lineaire de 0.10 a 0.65.
Pour verifier que le gain Chernoff tient au-dela, on elargit a 8 nouvelles instances avec `M`
(6 a 20), `p` plus eleve (0.20 a 0.76) et `n_runs` double (2000). À `p` fixé, les rayons de Hoeffding et de Chernoff décroissent en `1/sqrt(N)` ; le rayon de Bernstein comporte aussi une correction d'ordre `1/N`.

Rappel de la mécanique mesurée : `eps_H = sqrt(ln(2/eps_p) / (2N))` ne dépend pas de `p`, tandis que `eps_C = sqrt(2*ln(2/eps_p)*p*(1-p)/N)` se resserre quand `p` s'éloigne de 1/2. Avec `n_runs = 2000`, `N` vaut 8000 ou 10000, d'où des bornes Hoeffding de `0.01518` / `0.01358` dans cette cellule, contre `0.02480` / `0.02147` / `0.01921` pour `N = 3000 / 4000 / 5000` dans la cellule précédente.

In [3]:
// Illustration 2 — Hoeffding, Chernoff et Bernstein sur 8 nouvelles instances
// 8 nouvelles instances : plage elargie (M, p) et n_runs double
// Entrées et affichages des bornes cités de la sortie Python, cellule 6 ; aucun tirage cité.
var pythonNew = new (int M, int T, decimal P, int N, string H, string C, string B)[]
{
    ( 6, 4, 0.200m,  8000, "0.01518", "0.01215", "0.01230"),
    ( 8, 5, 0.280m, 10000, "0.01358", "0.01220", "0.01232"),
    (10, 4, 0.360m,  8000, "0.01518", "0.01458", "0.01473"),
    (12, 5, 0.440m, 10000, "0.01358", "0.01348", "0.01361"),
    (14, 4, 0.520m,  8000, "0.01518", "0.01517", "0.01533"),
    (16, 5, 0.600m, 10000, "0.01358", "0.01331", "0.01343"),
    (18, 4, 0.680m,  8000, "0.01518", "0.01417", "0.01432"),
    (20, 5, 0.760m, 10000, "0.01358", "0.01160", "0.01172")
};
const int newCount = 8;
const int newRuns = 2 * referenceRuns;
Require(newRuns == 2000 && pythonNew.Length == newCount, "Paramètres de l'extension différents de la référence Python.");

// On poursuit le même flux pseudo-aléatoire après les douze instances de référence.
var instancesNew = Enumerable.Range(0, newCount)
    .Select(i => Measure(6 + 2 * i, 4 + i % 2, 0.20 + 0.08 * i, newRuns, pythonNew[i]))
    .ToArray();
Require(instancesNew.Count(r => r.C < r.H) == newCount,
    "Le rayon de Chernoff doit être strictement inférieur à Hoeffding pour les huit nouvelles instances.");

Console.WriteLine(Inv($"=== Cellule 2 : Hoeffding vs Chernoff sur {newCount} nouvelles instances ==="));
Console.WriteLine(Inv($"Meme methode, plage elargie (n_runs={newRuns}) :"));
PrintTable(instancesNew, newCount);

=== Cellule 2 : Hoeffding vs Chernoff sur 8 nouvelles instances ===


Meme methode, plage elargie (n_runs=2000) :


   M   t      p      N   p_hat  hoeff(H)  chern(C) bernst(B)   flips  |p_hat-p|    3 sigma  Contrôle


   6   4  0.200   8000  0.2041   0.01518   0.01215   0.01230    1633   0.004125   0.013416  CONFORME


   8   5  0.280  10000  0.2863   0.01358   0.01220   0.01232    2863   0.006300   0.013470  CONFORME


  10   4  0.360   8000  0.3591   0.01518   0.01458   0.01473    2873   0.000875   0.016100  CONFORME


... (3 outputs left out)

  18   4  0.680   8000  0.6803   0.01518   0.01417   0.01432    5442   0.000250   0.015646  CONFORME


  20   5  0.760  10000  0.7612   0.01358   0.01160   0.01172    7612   0.001200   0.012812  CONFORME


Sous Hoeffding : 8/8 (attendu : 8/8).


Écarts maximaux aux références décimales : H=1.735E-018, C=1.735E-018, B=1.735E-018 (tolérance 1.0E-012).


### Lecture du résultat : le gain tient sur la plage élargie

Sur les 8 nouvelles instances (`M` de 6 à 20, `p` de 0.20 à 0.76, `n_runs = 2000`), `chern(C)` reste sous `hoeff(H)` sur toutes les lignes : `eps_C = 0.01160` contre `eps_H = 0.01358` pour `M=20, p=0.760, N=10000`. Sur cette dernière ligne, le tirage .NET donne `p_hat = 0.7612`, soit `|p_hat - p| = 0.001200`, en dessous du seuil `3 sigma`. La comparaison des rayons, elle, ne dépend pas du tirage.

## Mesure 3 : confrontation Chernoff vs Hoeffding globale

Agregation des 20 instances : on compare les ecarts `eps_H` (Hoeffding), `eps_C` (Chernoff)
et `eps_B` (Bernstein) au niveau de confiance nominal de 95 %. Le plus petit eps = le rayon le plus
serré = le plus **tight**.

Chaque `eps` est un rayon calculé, distinct de l'écart observé `|p_hat - p|`. Pour Chernoff, ce rayon vient de l'approximation KL quadratique et n'est pas une garantie à écart fini. La cellule suivante agrège les 20 instances, affiche les min/max des trois écarts et compte sur combien d'entre elles `eps_C < eps_H`, `eps_C = eps_H` ou `eps_H < eps_C`.

In [4]:
// Illustration 3 — Synthèse et attestation de parité sur 20 instances
const int expectedTotal = 20;
const int expectedChernoffStrict = 19;
const int expectedNotChernoffStrict = 1;
const int expectedEqual = 1;
const int expectedHoeffdingStrict = 0;
const int expectedHoeffdingCovered = 20;
const int expectedMonteCarloCovered = 20;

// Les extrema à cinq décimales sont cités de la synthèse Python (cellule 9).
// Avant arrondi, on calcule les extrema de référence par la voie décimale indépendante.
var allInstances = instancesRef.Concat(instancesNew).ToArray();
var allReferences = pythonRef.Concat(pythonNew)
    .Select(r => BoundsReference(r.N, r.P)).ToArray();
int total = allInstances.Length;
Require(total == expectedTotal && allReferences.Length == expectedTotal,
    Inv($"Effectif de la synthèse : C#={total}, références={allReferences.Length}, attendu={expectedTotal}."));

var extrema = new[]
{
    (Name: "Hoeffding", Min: allInstances.Min(r => r.H), Max: allInstances.Max(r => r.H),
     RefMin: allReferences.Min(r => r.H), RefMax: allReferences.Max(r => r.H), PythonMin: "0.01358", PythonMax: "0.02480"),
    (Name: "Chernoff", Min: allInstances.Min(r => r.C), Max: allInstances.Max(r => r.C),
     RefMin: allReferences.Min(r => r.C), RefMax: allReferences.Max(r => r.C), PythonMin: "0.01160", PythonMax: "0.02467"),
    (Name: "Bernstein", Min: allInstances.Min(r => r.B), Max: allInstances.Max(r => r.B),
     RefMin: allReferences.Min(r => r.B), RefMax: allReferences.Max(r => r.B), PythonMin: "0.01172", PythonMax: "0.02508")
};

Console.WriteLine(Inv($"=== Cellule 3 : Synthese Chernoff vs Hoeffding sur {total} instances ==="));
foreach (var e in extrema)
{
    double deltaMin = VerifyBound(e.Name, "Synthèse : minimum", e.Min, e.RefMin, e.PythonMin);
    double deltaMax = VerifyBound(e.Name, "Synthèse : maximum", e.Max, e.RefMax, e.PythonMax);
    Console.WriteLine(Inv($"Ecart {e.Name,-9} : min={e.Min:F5}, max={e.Max:F5}"));
    Console.WriteLine(Inv($"  Python cité : min={e.PythonMin}, max={e.PythonMax} ; écarts avant arrondi à la référence décimale : min={deltaMin:E3}, max={deltaMax:E3} — CONFORME"));
}

// Dans la sortie Python, le second effectif est le complément de C < H,
// même si son libellé dit H < C. L'égalité ne constitue pas une victoire.
int chernoffStrict = allInstances.Count(r => r.C < r.H);
int equal = allInstances.Count(r => r.C == r.H);
int hoeffdingStrict = allInstances.Count(r => r.H < r.C);
int notChernoffStrict = total - chernoffStrict;
Require(chernoffStrict == expectedChernoffStrict && notChernoffStrict == expectedNotChernoffStrict
    && equal == expectedEqual && hoeffdingStrict == expectedHoeffdingStrict
    && chernoffStrict + equal + hoeffdingStrict == total,
    Inv($"Ordre des rayons : C<H={chernoffStrict}, C=H={equal}, H<C={hoeffdingStrict}, complément de C<H={notChernoffStrict}."));

// Vérifier le gain maximal cité dans la lecture, défini par le facteur H/C.
var first = instancesRef[0];
double firstGain = first.H / first.C;
double maxGain = allInstances.Max(r => r.H / r.C);
Require(firstGain == maxGain && Inv($"{maxGain:F2}") == "1.67",
    Inv($"Gain maximal H/C : première instance={firstGain:G17}, maximum={maxGain:G17} ; attendu sur la première instance, à 1.67 après arrondi."));

Console.WriteLine();
Console.WriteLine("Comparaison hoeff vs chern :");
Console.WriteLine(Inv($"  - Chernoff plus tight (eps_C < eps_H) : {chernoffStrict}/{total} (Python cité : {expectedChernoffStrict}/{expectedTotal})"));
Console.WriteLine(Inv($"  - Égalité (eps_C = eps_H) : {equal}/{total} (attendu : {expectedEqual}/{expectedTotal})"));
Console.WriteLine(Inv($"  - Hoeffding plus tight (eps_H < eps_C) : {hoeffdingStrict}/{total} (attendu : {expectedHoeffdingStrict}/{expectedTotal})"));
Console.WriteLine(Inv($"  - Complément de eps_C < eps_H : {notChernoffStrict}/{total} (Python cité : {expectedNotChernoffStrict}/{expectedTotal})"));
Console.WriteLine(Inv($"  - Facteur maximal eps_H / eps_C : {maxGain:F2}, sur la première instance (M={first.M}, p={first.P:F3}, N={first.N})"));

// Attestation par lot : écarts déterministes et contrôles du tirage .NET.
Console.WriteLine();
Console.WriteLine("Attestation de parité par lot :");
foreach (var lot in new[] { (Name: "Référence", Rows: instancesRef), (Name: "Extension", Rows: instancesNew) })
{
    int count = lot.Rows.Length;
    int underH = lot.Rows.Count(r => r.Error <= r.H);
    int underMc = lot.Rows.Count(r => r.Error <= r.McLimit);
    Require(underH == count && underMc == count,
        Inv($"{lot.Name} : sous Hoeffding {underH}/{count}, sous le seuil Monte-Carlo {underMc}/{count} ; attendu {count}/{count} pour les deux contrôles."));
    Console.WriteLine(Inv($"{lot.Name} : écarts maximaux aux références décimales H={lot.Rows.Max(r => r.DeltaH):E3}, C={lot.Rows.Max(r => r.DeltaC):E3}, B={lot.Rows.Max(r => r.DeltaB):E3} (tolérance {boundTolerance:E1}) ; sous Hoeffding {underH}/{count} ; sous le seuil Monte-Carlo {underMc}/{count} — CONFORME"));
}
int hoeffdingCovered = allInstances.Count(r => r.Error <= r.H);
int monteCarloCovered = allInstances.Count(r => r.Error <= r.McLimit);
Require(hoeffdingCovered == expectedHoeffdingCovered && monteCarloCovered == expectedMonteCarloCovered,
    Inv($"Couverture globale : sous Hoeffding {hoeffdingCovered}/{total}, sous le seuil Monte-Carlo {monteCarloCovered}/{total} ; attendus {expectedHoeffdingCovered}/{expectedTotal} et {expectedMonteCarloCovered}/{expectedTotal}."));
Console.WriteLine(Inv($"Total : sous Hoeffding {hoeffdingCovered}/{total} (attendu : {expectedHoeffdingCovered}/{expectedTotal}) ; sous le seuil Monte-Carlo {monteCarloCovered}/{total} (attendu : {expectedMonteCarloCovered}/{expectedTotal}) — CONFORME"));

=== Cellule 3 : Synthese Chernoff vs Hoeffding sur 20 instances ===


Ecart Hoeffding : min=0.01358, max=0.02480


  Python cité : min=0.01358, max=0.02480 ; écarts avant arrondi à la référence décimale : min=0.000E+000, max=3.469E-018 — CONFORME


Ecart Chernoff  : min=0.01160, max=0.02467


  Python cité : min=0.01160, max=0.02467 ; écarts avant arrondi à la référence décimale : min=1.735E-018, max=0.000E+000 — CONFORME


Ecart Bernstein : min=0.01172, max=0.02508


... (7 outputs left out)

Attestation de parité par lot :


Référence : écarts maximaux aux références décimales H=3.469E-018, C=3.469E-018, B=3.469E-018 (tolérance 1.0E-012) ; sous Hoeffding 12/12 ; sous le seuil Monte-Carlo 12/12 — CONFORME


Extension : écarts maximaux aux références décimales H=1.735E-018, C=1.735E-018, B=1.735E-018 (tolérance 1.0E-012) ; sous Hoeffding 8/8 ; sous le seuil Monte-Carlo 8/8 — CONFORME


Total : sous Hoeffding 20/20 (attendu : 20/20) ; sous le seuil Monte-Carlo 20/20 (attendu : 20/20) — CONFORME


## Interpretation

**Resultat** : sur 19/20 instances BSC, le rayon de Chernoff (approximation KL quadratique) est plus tight que Hoeffding
(eps_C < eps_H) ; la 20e (p=0.50) donne des ecarts numeriquement egaux (eps_C = eps_H),
ce qui est attendu theoriquement (a p=1/2, 2*p*(1-p) = 1/2, l'exposant
-N*eps^2 / (2*p*(1-p)) devient -2*N*eps^2 et l'approximation de Chernoff se reduit exactement
a la forme Hoeffding). Le gain vient de ce que Chernoff exploite la **vraie variance** `p(1-p)`
de la Bernoulli, alors que Hoeffding utilise le majorant grossier `1/4` (pire cas p=1/2).
Quand p est proche de 0 ou de 1, la variance reelle est beaucoup plus petite que 1/4, et
le gain de Chernoff augmente (jusqu'a un facteur 1.67 dans le tableau C#, pour p=0.1, N=3000 :
eps_H = 0.02480 vs eps_C = 0.01488).

**Bernstein** exploite la variance `p(1-p)` comme Chernoff, mais reste conservative pour
les ecarts `eps` grands (terme `eps/3` au denominateur). Elle est particulierement adaptee
quand on veut une borne de queue bilaterale pour des variables bornées sans connaître leur loi exacte.
Son rayon ne se situe pas toujours entre les deux autres : pour `p = 0.550` et `N = 3000`, le tableau donne `eps_B = 0.02508` contre `eps_H = 0.02480`.

**Implication pour la detection MIMO** : Lean-19 etablissait que la detection tient avec
Hoeffding (borne conservative, distribution-free). Lean-19b compare les rayons quand on connait la distribution exacte (Bernoulli dans le cas BSC). Pour
le cas MIMO general (non-BSC), Bernstein peut être un compromis si les hypothèses de la borne sont satisfaites.

**Limites** : (a) l'approximation KL quadratique est locale ; pour la queue supérieure,
la majoration de Chernoff non approximée est `exp(-N * KL(p+delta || p))` avec KL la divergence de
Kullback-Leibler. (b) Pour les distributions non-Bernoulli, cette formule spécifique ne s'applique pas
directement. (c) Le gain de Chernoff depend du modele ;
pour p=1/2, le rayon quadratique et celui de Hoeffding sont numeriquement egaux (les deux `eps` valent `0.01921` pour l'unique instance à `p = 0.50`, ligne `M=12`).

## Exercices

### Exercice 1 : fonction generatrice des moments de la Bernoulli

Implementer `mgf_bernoulli(t, p)` qui retourne la mgf d'une Bernoulli(p) evaluee en `t`. Vérifier sa normalisation et le lien entre sa dérivée à l'origine et l'espérance.

In [5]:
double? mgf_bernoulli(double t, double p)
{
    // TODO etudiant
    return null;
}

// Test : propriétés de la MGF à l'origine.
if (!mgf_bernoulli(0.0, 0.3).HasValue)
    Console.WriteLine("Exercice 1 a completer");

Exercice 1 a completer


### Exercice 2 : borne de Chernoff (approx KL) pour un ecart donne

Implementer `chernoff_eps_exo(N, p, eps_p=0.05)` qui retourne l'écart `eps` pour lequel l'expression bilatérale de Chernoff (approximation KL quadratique) atteint `eps_p`. Vérifier le résultat pour `N=1000, p=0.1, eps_p=0.05`.

In [6]:
double? chernoff_eps_exo(int N, double p, double eps_p = 0.05)
{
    // TODO etudiant
    return null;
}

if (!chernoff_eps_exo(1000, 0.1).HasValue)
    Console.WriteLine("Exercice 2 a completer");

Exercice 2 a completer


### Exercice 3 : comparer Chernoff et Hoeffding sur une plage parametrique

Implementer `ratio_chernoff_hoeffding(N, p, eps_p)`. Tracer (ou afficher en table) le ratio `eps_H / eps_C` en fonction de `p` pour `N=5000`
et `eps_p=0.05`, sur la grille `p = 0.05, 0.1, 0.2, 0.3, 0.4, 0.5`. Décrire l'évolution du gain sur cette grille.

In [7]:
double? ratio_chernoff_hoeffding(int N, double p, double eps_p = 0.05)
{
    // TODO etudiant
    return null;
}

// Paramètres de l'en-tête affiché par le carnet Python ; le stub ne produit aucun ratio.
const int ratioN = 5000;
const double ratioEpsP = epsP;
string ratioHeader = Inv($"Ratio eps_H / eps_C pour N={ratioN}, eps_p={ratioEpsP:F2} :");
Require(ratioHeader == "Ratio eps_H / eps_C pour N=5000, eps_p=0.05 :",
    "L'en-tête de l'exercice diffère de la sortie Python de référence.");
Console.WriteLine(ratioHeader);

Ratio eps_H / eps_C pour N=5000, eps_p=0.05 :


## Conclusion et references

Sur 19/20 instances BSC, le rayon de Chernoff dans l'approximation KL quadratique est plus tight que celui de Hoeffding (egalite numerique a `p=0.50` pour l'autre instance). Le gain provient de l'exploitation de la variance reelle `p(1-p)` au lieu du majorant Hoeffding `1/4`. Bernstein exploite aussi la variance mais ajoute un terme correctif ; son rayon n'est pas toujours intermédiaire.

D'après le [carnet Python](Lean-19b-Borne-Chernoff-Python.ipynb), les carnets MIMO post-capstone exécutent la détection sur le canal MIMO du lac `mimo_lean`. Cette information est citée de la source ; cette exécution n'a pas été vérifiée ici.

**References** :

- [Lean-19 — Hoeffding](Lean-19-Detection-MIMO-Index-Python.ipynb) -- le carnet pere, etablit la borne Hoeffding empiriquement sur 20 instances (20/20 sous la borne, résultat cité de Lean-19).
- [Lean-20 — Capstone](Lean-20-Capstone-Digestions-Tao-Python.ipynb) -- la capstone de la serie, qui digere les techniques de Terence Tao sur les flots de Ricci.
- [MIMO-01 -- Descente Budget](MIMO/MIMO-01-Descente-Budget-Lean-Python.ipynb) -- la borne de budget pour la detection MIMO, qui consomme Lean-19 et Lean-19b.
- [MIMO/README.md](MIMO/README.md) -- la descente MIMO-01..04, qui consomme Lean-19b comme etalon de borne de queue.

**Bilan chiffré de la synthèse C#** : Chernoff donne un rayon strictement plus petit dans 19/20 instances, avec une égalité et aucune victoire stricte de Hoeffding. Le `1/20` de la sortie Python citée sous « Hoeffding plus tight » est le complément de `eps_C < eps_H`, pas le compte des cas `eps_H < eps_C`. Les extrema recalculés sont `min=0.01160, max=0.02467` pour Chernoff et `min=0.01358, max=0.02480` pour Hoeffding. Le gain maximal observé vient de la première instance (`p=0.100, N=3000`) : `eps_H = 0.02480` vs `eps_C = 0.01488`.

> Provenance : [Version Python](Lean-19b-Borne-Chernoff-Python.ipynb).